<a href="https://colab.research.google.com/github/VeenusDadri/training/blob/master/basic_pyspark/Tungsten_%26_Catalyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("TungstenOptimization").getOrCreate()

data = [("Apple", "Fruit", 52),
        ("Banana", "Fruit", 89),
        ("Carrot", "Vegetable", 41),
        ("Tomato", "Vegetable", 18),
        ("Mango", "Fruit", 60),
        ("Broccoli", "Vegetable", 55)]

columns = ["Name", "Type", "Calories"]

df = spark.createDataFrame(data, columns)

optimized_df = df.withColumn("Calories_Doubled", col("Calories") * 2)

optimized_df.show()

+--------+---------+--------+----------------+
|    Name|     Type|Calories|Calories_Doubled|
+--------+---------+--------+----------------+
|   Apple|    Fruit|      52|             104|
|  Banana|    Fruit|      89|             178|
|  Carrot|Vegetable|      41|              82|
|  Tomato|Vegetable|      18|              36|
|   Mango|    Fruit|      60|             120|
|Broccoli|Vegetable|      55|             110|
+--------+---------+--------+----------------+



In [2]:


df.createOrReplaceTempView("foods")

optimized_query = spark.sql("SELECT Name, Calories FROM foods WHERE Calories > 50")

optimized_query.show()

optimized_query.explain(mode="formatted")

+--------+--------+
|    Name|Calories|
+--------+--------+
|   Apple|      52|
|  Banana|      89|
|   Mango|      60|
|Broccoli|      55|
+--------+--------+

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [Name#0, Type#1, Calories#2L]
Arguments: [Name#0, Type#1, Calories#2L], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [3]: [Name#0, Type#1, Calories#2L]
Condition : (isnotnull(Calories#2L) AND (Calories#2L > 50))

(3) Project [codegen id : 1]
Output [2]: [Name#0, Calories#2L]
Input [3]: [Name#0, Type#1, Calories#2L]




In [3]:


df_transformed = df.withColumn("Calories_Doubled", col("Calories") * 2)

df_transformed.explain(mode="formatted")
df_transformed.show()

== Physical Plan ==
* Project (2)
+- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [Name#0, Type#1, Calories#2L]
Arguments: [Name#0, Type#1, Calories#2L], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [4]: [Name#0, Type#1, Calories#2L, (Calories#2L * 2) AS Calories_Doubled#39L]
Input [3]: [Name#0, Type#1, Calories#2L]


+--------+---------+--------+----------------+
|    Name|     Type|Calories|Calories_Doubled|
+--------+---------+--------+----------------+
|   Apple|    Fruit|      52|             104|
|  Banana|    Fruit|      89|             178|
|  Carrot|Vegetable|      41|              82|
|  Tomato|Vegetable|      18|              36|
|   Mango|    Fruit|      60|             120|
|Broccoli|Vegetable|      55|             110|
+--------+---------+--------+----------------+



In [4]:
df.createOrReplaceTempView("foods")

optimized_query = spark.sql("SELECT Name FROM foods WHERE Calories > 50")

optimized_query.explain(mode="formatted")
optimized_query.show()

== Physical Plan ==
* Project (3)
+- * Filter (2)
   +- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [3]: [Name#0, Type#1, Calories#2L]
Arguments: [Name#0, Type#1, Calories#2L], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter [codegen id : 1]
Input [3]: [Name#0, Type#1, Calories#2L]
Condition : (isnotnull(Calories#2L) AND (Calories#2L > 50))

(3) Project [codegen id : 1]
Output [1]: [Name#0]
Input [3]: [Name#0, Type#1, Calories#2L]


+--------+
|    Name|
+--------+
|   Apple|
|  Banana|
|   Mango|
|Broccoli|
+--------+



In [5]:
from pyspark.sql.functions import broadcast

price_data = [("Apple", 1.2), ("Banana", 0.5), ("Carrot", 0.8), ("Tomato", 1.5), ("Mango", 2.0), ("Broccoli", 1.8)]
price_columns = ["Name", "Price_per_kg"]

df_price = spark.createDataFrame(price_data, price_columns)

joined_df = df.join(broadcast(df_price), "Name", "inner")

joined_df.explain(mode="formatted")
joined_df.show()

== Physical Plan ==
AdaptiveSparkPlan (8)
+- Project (7)
   +- BroadcastHashJoin Inner BuildRight (6)
      :- Filter (2)
      :  +- Scan ExistingRDD (1)
      +- BroadcastExchange (5)
         +- Filter (4)
            +- Scan ExistingRDD (3)


(1) Scan ExistingRDD
Output [3]: [Name#0, Type#1, Calories#2L]
Arguments: [Name#0, Type#1, Calories#2L], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [3]: [Name#0, Type#1, Calories#2L]
Condition : isnotnull(Name#0)

(3) Scan ExistingRDD
Output [2]: [Name#67, Price_per_kg#68]
Arguments: [Name#67, Price_per_kg#68], MapPartitionsRDD[17] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(4) Filter
Input [2]: [Name#67, Price_per_kg#68]
Condition : isnotnull(Name#67)

(5) BroadcastExchange
Input [2]: [Name#67, Price_per_kg#68]
Arguments: HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=